In [ ]:
from promoter_protein_clip.model_transformer import SequenceCLIPTransformer
import torch
from huggingface_hub import hf_hub_download

repo_id = "dufaultc/contrastive-promoter-protein-pretraining"
filename = "C3P_100M.pth"

promoter = "ACTCGCTATGCGCTATATCTCTATACGGTTGCCGCGCGGTGTCCCCCGGTAAACTCGCTATGCGCTATATCTCTATACGGTTGCCGCGCGGTGTCCCCCGGTAAACTCGCTATGCGCTATATCTCTATACGGTTGCCGCGCGGTGTCCCCCGGTAA"
checkpoint_path = hf_hub_download(repo_id=repo_id, filename=filename )

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")    
checkpoint= torch.load(checkpoint_path, map_location=device, weights_only=False)    
model = SequenceCLIPTransformer(
    protein_model_name=checkpoint['args'].protein_model,
    promoter_embedding_dim=checkpoint['args'].promoter_embedding_dim,
    depth=checkpoint['args'].transformer_depth,
    num_heads=checkpoint['args'].transformer_num_heads,
    max_promoter_length=checkpoint['args'].max_promoter_length,
    projection_dim=checkpoint['args'].projection_dim,       
 ).to(device)
model = torch.compile(model)
model.load_state_dict(checkpoint['model_state_dict'])  
model.eval()
batch_inputs = model.promoter_tokenizer(
    [promoter],
    padding=True,
    truncation=True,
    return_tensors="pt",
)
with torch.no_grad():
    embeddings = (
        model.promoter_encoder(batch_inputs.to(device)).cpu().numpy()
    )   

Loading weights: 100%|██████████| 486/486 [00:00<00:00, 13103.58it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t30_150M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
embeddings

array([[ 0.14442241,  0.47437054, -0.9122608 , ..., -1.4887502 ,
         1.3440311 , -0.8058226 ]], shape=(1, 1024), dtype=float32)